In [1]:
# Cài đặt các thư viện cần thiết nếu máy bạn chưa có
!pip install transformers peft torch

  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 1.4 MB/s eta 0:00:0000:0100:01
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 2.6 MB/s eta 0:00:0000:0100:01
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 5.5 MB/s eta 0:00:00
Using cac

In [2]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Đường dẫn tới thư mục chứa file LoRA bạn vừa tải về
ADAPTER_PATH = "./codebert-vuln-lora-final" 
BASE_MODEL = "microsoft/codebert-base"

# Setup Device (Tự động nhận diện GPU nếu có, không thì chạy CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Đang sử dụng thiết bị: {device}")

LABEL2ID = {"Safe": 0, "Vulnerable": 1}
ID2LABEL = {0: "Safe", 1: "Vulnerable"}
MAX_TOKENS = 512

ModuleNotFoundError: Could not import module 'AutoTokenizer'. Are this object's requirements defined correctly?

In [ ]:
# Trích xuất tên hàm và tên contract để điền vào Prompt
def extract_fn_and_contract(code: str):
    fn_match = re.search(r'function\s+(\w+)\s*\(', code)
    contract_match = re.search(r'contract\s+(\w+)', code)
    fn_name = fn_match.group(1) if fn_match else "unknownFunction"
    contract_name = contract_match.group(1) if contract_match else "UnknownContract"
    return fn_name, contract_name

# 5 Biến thể Prompt (Giữ nguyên như lúc Training)
PROMPT_VARIANTS = [
    {
        "a": "Devise a label name suitable for categorizing items as either vulnerable or safe.",
        "b": "Please review the code. Please find out if it is vulnerable.",
        "c": "The function {fn_name} from the contract {contract_name}.",
    },
    {
        "a": "Suggest a label designation that clearly identifies an item's status as either vulnerable or safe.",
        "b": "Inspect the following Solidity code. Determine if there are any vulnerabilities present.",
        "c": "Observe the method {fn_name} within the smart contract {contract_name}.",
    },
    {
        "a": "Invent a naming label that aptly segregates items into vulnerable or safe classifications.",
        "b": "Examine this Solidity script. Identify any potential security risks.",
        "c": "Review the function {fn_name} in the blockchain contract {contract_name}.",
    },
    {
        "a": "Formulate a label descriptor that bifurcates objects into categories of vulnerable and safe.",
        "b": "Please assess the provided Solidity code for any security vulnerabilities.",
        "c": "Check the procedure {fn_name} in the digital contract {contract_name}.",
    },
    {
        "a": "Propose a label nomenclature that aptly differentiates between vulnerable and safe states.",
        "b": "Evaluate the given Solidity function. Are there any security flaws?",
        "c": "Inspect the subroutine {fn_name} from the decentralized contract {contract_name}.",
    }
]

def build_prompt_text(code: str, variant_idx: int) -> str:
    v = PROMPT_VARIANTS[variant_idx]
    fn_name, contract_name = extract_fn_and_contract(code)
    c_filled = v["c"].format(fn_name=fn_name, contract_name=contract_name)
    return f"{v['a']} {v['b']} {c_filled}"

In [ ]:
print("⏳ Đang tải Tokenizer...")
# Tokenizer đã được lưu cùng với adapter lúc train
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

print("⏳ Đang tải Base Model...")
# 1. Load mô hình gốc (chỉ cần load kiến trúc Sequence Classification)
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    trust_remote_code=True,
)

print("⏳ Đang gắn trọng số LoRA...")
# 2. Gắn Adapter LoRA vào mô hình gốc
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.to(device)
model.eval() # Chuyển sang chế độ dự đoán

print("✅ Hệ thống quét lỗ hổng đã sẵn sàng!")

⏳ Đang tải Tokenizer...
⏳ Đang tải Base Model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 50450.48it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


⏳ Đang gắn trọng số LoRA...
✅ Hệ thống quét lỗ hổng đã sẵn sàng!


In [ ]:
def predict_single(code: str, variant_idx: int) -> str:
    """Dự đoán với 1 prompt cụ thể"""
    prompt_text = build_prompt_text(code, variant_idx)
    encoding = tokenizer(
        prompt_text,
        code,
        max_length=MAX_TOKENS,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
    
    pred_id = torch.argmax(outputs.logits, dim=-1).item()
    return ID2LABEL[pred_id]

def scan_smart_contract(code: str):
    """Quét toàn diện bằng Majority Voting (5 Prompts)"""
    print("🔍 Đang quét mã nguồn...")
    votes = []
    
    # Chạy qua cả 5 biến thể prompt
    for i in range(5):
        pred = predict_single(code, i)
        votes.append(pred)
        
    n_vuln = votes.count("Vulnerable")
    n_safe = 5 - n_vuln
    
    # Quyết định cuối cùng (>= 3 phiếu là Vulnerable)
    final_result = "Vulnerable" if n_vuln >= 3 else "Safe"
    
    print("-" * 40)
    print(f"Kết quả bỏ phiếu (Vulnerable/Safe): {n_vuln}/{n_safe}")
    print(f"Kết luận cuối cùng: {'CÓ LỖ HỔNG (VULNERABLE)' if final_result == 'Vulnerable' else 'AN TOÀN (SAFE)'}")
    print("-" * 40)
    
    return final_result

In [ ]:
# Thử nghiệm với một đoạn mã Smart Contract
sample_code = """
function fetchPrice() external returns (uint); }
"""

# Chạy hệ thống
result = scan_smart_contract(sample_code)

🔍 Đang quét mã nguồn...
----------------------------------------
Kết quả bỏ phiếu (Vulnerable/Safe): 0/5
Kết luận cuối cùng: AN TOÀN (SAFE)
----------------------------------------
